# Validation Pipeline
AgentTrust Validation Pipeline Notebook

In [1]:
# Setup
print('Validation Pipeline Ready')

Validation Pipeline Ready


## Cell 1 — Install the SDK

In [2]:
import sys
!uv pip install "agentrust-py[embedded,retry]" --python {sys.executable}

Checked 1 package in 49ms


## Cell 2 — Start the embedded gateway + imports

`embed_gateway()` spins up an in-process SQLite gateway (no external service needed).  
Must be called **before** any `@harness`-decorated function is invoked.

In [3]:
import os
from agentrust_sdk import harness, embed_gateway, BlockedError

# Start in-process SQLite gateway (dev mode — no external service needed)
gw = embed_gateway()
print("Gateway started:", gw)

Gateway started: <agentrust_sdk.embedded.EmbeddedGateway object at 0x108574f80>


## Cell 3 — Define the agent

A simple **text-summariser agent**: counts words, sentences, and returns the most frequent word.  
The `@harness` decorator intercepts the return value and sends it to the gateway for governance before the caller sees it.

In [4]:
import re
from collections import Counter

@harness
def summarise_agent(user: str, input: str) -> dict:
    """
    Reads a block of text and returns basic stats + the top word.
    Governance runs automatically after this returns.
    """
    words = re.findall(r"\b\w+\b", input.lower())
    sentences = [s.strip() for s in re.split(r"[.!?]", input) if s.strip()]
    top_word, top_count = Counter(words).most_common(1)[0] if words else ("", 0)

    return {
        "word_count":     len(words),
        "sentence_count": len(sentences),
        "top_word":       top_word,
        "top_word_count": top_count,
        "summary":        f"{len(words)} words across {len(sentences)} sentence(s). Most common: '{top_word}' ({top_count}x).",
    }

print("Agent defined:", summarise_agent)

Agent defined: <function summarise_agent at 0x108a98220>


## Cell 4 — Run the agent (normal path)

Expected: governance approves, result dict is returned.

In [11]:
TEXT = (
    "The quick brown fox jumps over the lazy dog. "
    "The dog did not seem to mind. "
    "The fox ran away quickly."
)

try:
    result = summarise_agent(user="alice", input=TEXT)
    print("RESULT:", result)
except BlockedError as e:
    print("BLOCKED by governance:", e)

[AgentTrust] Gateway unreachable (fail-open): Client error '401 Unauthorized' for url 'http://127.0.0.1:8765/v1/runtime/validate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


RESULT: {'word_count': 21, 'sentence_count': 3, 'top_word': 'the', 'top_word_count': 4, 'summary': "21 words across 3 sentence(s). Most common: 'the' (4x)."}


## Cell 5 — Probe the SDK's `ValidateResponse` directly

Use `AgentTrustClient.validate()` to see what the gateway actually returns — decision, risk tier, scores — without the decorator hiding it.

In [ ]:
from agentrust_sdk import AgentTrustClient

client = AgentTrustClient()

sample_output = {
    "word_count": 18,
    "sentence_count": 3,
    "top_word": "the",
    "top_word_count": 4,
    "summary": "18 words across 3 sentence(s). Most common: 'the' (4x).",
}

r = client.validate(
    agent_id="summarise-agent",
    user="alice",
    input=TEXT,
    output=sample_output,
    framework="notebook",
)

print("envelope_id  :", r.envelope_id)
print("approved     :", r.approved)
print("blocked      :", r.blocked)
print("needs_review :", r.needs_review)
print()
print("decision.outcome       :", r.decision.outcome)
print("decision.reason        :", r.decision.reason)
print("decision.policy_version:", r.decision.policy_version)
print()
print("risk.tier  :", r.risk.tier)
print("risk.score :", r.risk.score)
print("risk.reason:", r.risk.reason)
print()
print("validation.policy_score    :", r.validation.policy_score)
print("validation.final_confidence:", r.validation.final_confidence)
print("validation.failures        :", r.validation.failures)

## Cell 6 — Kill-switch: disable governance at runtime

Set `AGENTRUST_ENABLED=false` → `@harness` becomes a transparent no-op.  
The agent still runs; governance is skipped entirely.

In [ ]:
os.environ["AGENTRUST_ENABLED"] = "false"

# Re-decorate so the kill-switch is picked up
# (In a real app you'd restart the process; in a notebook we re-apply the decorator)
@harness
def summarise_agent_disabled(user: str, input: str) -> dict:
    words = re.findall(r"\b\w+\b", input.lower())
    sentences = [s.strip() for s in re.split(r"[.!?]", input) if s.strip()]
    top_word, top_count = Counter(words).most_common(1)[0] if words else ("", 0)
    return {
        "word_count": len(words),
        "sentence_count": len(sentences),
        "top_word": top_word,
        "top_word_count": top_count,
        "summary": f"{len(words)} words, most common: '{top_word}'.",
    }

result_disabled = summarise_agent_disabled(user="alice", input=TEXT)
print("Result with governance OFF:", result_disabled)

# Re-enable for subsequent cells
os.environ["AGENTRUST_ENABLED"] = "true"
print("Governance re-enabled.")

## Cell 7 — Failure mode: `closed`

With `AGENTRUST_FAILURE_MODE=closed`, if the gateway is unreachable the SDK raises `GatewayUnavailableError` instead of silently approving.  
Here we simulate it by pointing at a dead URL.

In [ ]:
from agentrust_sdk import GatewayUnavailableError

os.environ["AGENTRUST_FAILURE_MODE"] = "closed"
os.environ["AGENTRUST_GATEWAY_URL"]  = "http://localhost:9999"  # dead port
os.environ["AGENTRUST_TIMEOUT_SEC"]  = "2"                      # fail fast

@harness
def summarise_agent_closed(user: str, input: str) -> dict:
    words = re.findall(r"\b\w+\b", input.lower())
    return {"word_count": len(words), "summary": f"{len(words)} words."}

try:
    summarise_agent_closed(user="alice", input=TEXT)
except GatewayUnavailableError as e:
    print("GatewayUnavailableError (expected in closed mode):", e)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
finally:
    # Restore env
    os.environ["AGENTRUST_FAILURE_MODE"] = "open"
    os.environ.pop("AGENTRUST_GATEWAY_URL", None)
    os.environ.pop("AGENTRUST_TIMEOUT_SEC", None)
    print("Env restored.")

# Price-Comparison Agent — Steps 0–6

End-to-end LangChain agent that fetches two product pages, extracts prices, compares them,
and (in Step 6) validates the output through AgentTrust governance.

Each phase prints timing and structured metrics so you can diff unmonitored vs governed runs.

## Step 0 — Install Dependencies

In [ ]:
import sys
# Core LangChain stack + utilities
!{sys.executable} -m pip install \
    langchain langchain-community langchain-openai \
    requests beautifulsoup4 python-dotenv pandas \
    pyyaml -q
print("Dependencies installed ✓")

## Step 1a — Mock Price Server

Spins up a local HTTP server (port 18080) so the agent can make real HTTP requests without
hitting live sites. Site A serves `$49.99`, Site B serves `$44.95`.

In [ ]:
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler

SITE_A_HTML = """<!DOCTYPE html>
<html><head><title>Site A — Widget</title></head>
<body>
  <h1>Super Widget — Site A</h1>
  <span class="price">$49.99</span>
  <p>Free shipping on orders over $30</p>
</body></html>"""

SITE_B_HTML = """<!DOCTYPE html>
<html><head><title>Site B — Widget</title></head>
<body>
  <h1>Super Widget — Site B</h1>
  <span class="price">$44.95</span>
  <p>Premium quality, lowest price</p>
</body></html>"""

class MockPriceHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args): pass  # suppress server logs

    def do_GET(self):
        if self.path == "/site-a/product":
            body = SITE_A_HTML.encode()
        elif self.path == "/site-b/product":
            body = SITE_B_HTML.encode()
        else:
            body = b"<html><body>404 Not found</body></html>"
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.end_headers()
        self.wfile.write(body)

_mock_server = HTTPServer(("127.0.0.1", 18080), MockPriceHandler)
threading.Thread(target=_mock_server.serve_forever, daemon=True).start()

print("Mock price server running on http://127.0.0.1:18080")
print("  Site A: http://127.0.0.1:18080/site-a/product  →  $49.99")
print("  Site B: http://127.0.0.1:18080/site-b/product  →  $44.95")

## Step 1b — Tool Definitions

Three standalone functions — build and test these before wiring into LangChain (per plan).

In [ ]:
import re
import time
import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup

APPROVED_DOMAINS = {"127.0.0.1:18080"}

def fetch_html(url: str) -> dict:
    """Fetch raw HTML from an approved domain. Returns status, html, size, elapsed_sec."""
    domain = urlparse(url).netloc
    if domain not in APPROVED_DOMAINS:
        raise ValueError(f"Domain not approved: {domain}")
    t0 = time.perf_counter()
    resp = requests.get(url, timeout=10)
    elapsed = round(time.perf_counter() - t0, 4)
    return {
        "url": url,
        "status": resp.status_code,
        "html": resp.text,
        "size": len(resp.text),
        "elapsed_sec": elapsed,
    }

def extract_price(html: str) -> dict:
    """Parse a price from HTML. Returns price float and diagnostic match list."""
    soup = BeautifulSoup(html, "html.parser")
    matches = re.findall(r"\$\s?(\d+\.\d{2})", soup.get_text())
    return {
        "matches_found": len(matches),
        "price": float(matches[0]) if len(matches) == 1 else None,
        "raw_matches": matches,
    }

def compare_prices(price_a: float, price_b: float) -> dict:
    """Compare two prices; return which is cheaper and the absolute difference."""
    if price_a is None or price_b is None:
        return {"error": "Missing price — cannot compare"}
    cheaper = "A" if price_a < price_b else "B"
    return {
        "cheaper_site": cheaper,
        "price_a": price_a,
        "price_b": price_b,
        "difference": round(abs(price_a - price_b), 2),
    }

print("Tools defined: fetch_html, extract_price, compare_prices ✓")

## Step 1c — Standalone Tool Validation + Phase Metrics

Run each tool independently and capture per-phase timing.
**Do not proceed to Step 2 until this produces sane output.**

In [ ]:
import time as _t

print("=" * 62)
print("STEP 1 VALIDATION — Standalone Tool Tests")
print("=" * 62)

phase_metrics: dict = {}

# ── fetch_html ──────────────────────────────────────────────
t0 = _t.perf_counter()
fetch_a = fetch_html("http://127.0.0.1:18080/site-a/product")
fetch_b = fetch_html("http://127.0.0.1:18080/site-b/product")
fetch_total = round(_t.perf_counter() - t0, 4)
phase_metrics["fetch_html"] = {
    "calls": 2,
    "total_duration_sec": fetch_total,
    "status_a": fetch_a["status"],
    "status_b": fetch_b["status"],
    "size_a_bytes": fetch_a["size"],
    "size_b_bytes": fetch_b["size"],
}
print(f"\n[fetch_html]")
print(f"  Site A → HTTP {fetch_a['status']} | {fetch_a['size']} bytes | {fetch_a['elapsed_sec']}s")
print(f"  Site B → HTTP {fetch_b['status']} | {fetch_b['size']} bytes | {fetch_b['elapsed_sec']}s")
print(f"  Phase total: {fetch_total}s")

# ── extract_price ────────────────────────────────────────────
t0 = _t.perf_counter()
price_a = extract_price(fetch_a["html"])
price_b = extract_price(fetch_b["html"])
extract_total = round(_t.perf_counter() - t0, 6)
phase_metrics["extract_price"] = {
    "calls": 2,
    "total_duration_sec": extract_total,
    "price_a": price_a["price"],
    "price_b": price_b["price"],
    "matches_a": price_a["raw_matches"],
    "matches_b": price_b["raw_matches"],
}
print(f"\n[extract_price]")
print(f"  Site A → ${price_a['price']}  (raw matches: {price_a['raw_matches']})")
print(f"  Site B → ${price_b['price']}  (raw matches: {price_b['raw_matches']})")
print(f"  Phase total: {extract_total}s")

# ── compare_prices ───────────────────────────────────────────
t0 = _t.perf_counter()
comparison = compare_prices(price_a["price"], price_b["price"])
compare_total = round(_t.perf_counter() - t0, 6)
phase_metrics["compare_prices"] = {
    "calls": 1,
    "total_duration_sec": compare_total,
    **comparison,
}
print(f"\n[compare_prices]")
print(f"  Cheaper site : {comparison['cheaper_site']}")
print(f"  Price A      : ${comparison['price_a']}")
print(f"  Price B      : ${comparison['price_b']}")
print(f"  Difference   : ${comparison['difference']}")
print(f"  Phase total  : {compare_total}s")

# ── summary table ────────────────────────────────────────────
print("\n" + "=" * 62)
print("PHASE METRICS SUMMARY — Step 1")
print("=" * 62)
print(f"  {'Phase':<20} {'Calls':>6}  {'Duration (s)':>14}")
print(f"  {'-'*20}  {'-'*6}  {'-'*14}")
for tool, m in phase_metrics.items():
    print(f"  {tool:<20} {m['calls']:>6}  {m['total_duration_sec']:>14.6f}")

print("\n✓ All standalone tools validated — safe to proceed to Step 2")

## Step 2 — Wrap as LangChain StructuredTools

Each function gets a Pydantic schema so the LLM can populate arguments correctly.

In [ ]:
from langchain.tools import StructuredTool
from pydantic import BaseModel
from typing import Optional

class FetchInput(BaseModel):
    url: str

class ExtractInput(BaseModel):
    html: str

class CompareInput(BaseModel):
    price_a: Optional[float] = None
    price_b: Optional[float] = None

fetch_tool = StructuredTool.from_function(
    func=fetch_html,
    name="fetch_html",
    description="Fetch HTML from an approved product page URL. Returns status, html, size, elapsed_sec.",
    args_schema=FetchInput,
)
extract_tool = StructuredTool.from_function(
    func=extract_price,
    name="extract_price",
    description="Extract a numeric price from fetched HTML. Returns price as a float.",
    args_schema=ExtractInput,
)
compare_tool = StructuredTool.from_function(
    func=compare_prices,
    name="compare_prices",
    description="Compare two prices and return which site is cheaper with the dollar difference.",
    args_schema=CompareInput,
)

tools = [fetch_tool, extract_tool, compare_tool]

print("LangChain StructuredTools created ✓")
for t in tools:
    schema = t.args_schema.model_json_schema()
    params = list(schema.get("properties", {}).keys())
    print(f"  {t.name:20s}  params: {params}")

## Step 3 — Execution Trace Callback Handler

Captures every tool start/end with a `correlation_id` that links execution events to
governance rows in Step 6. Also records per-tool elapsed time and LLM token usage.

In [ ]:
import json, time, uuid, os
from collections import defaultdict
from langchain.callbacks.base import BaseCallbackHandler

os.makedirs("logs", exist_ok=True)

class ExecutionTraceHandler(BaseCallbackHandler):
    """Writes a JSONL execution trace.  Every tool-start/end pair shares a
    correlation_id so the governance trace can be joined on that key."""

    def __init__(self, run_id: str, log_path: str = "logs/execution_trace.jsonl"):
        self.run_id   = run_id
        self.log_path = log_path
        self._pending: dict      = {}   # f"{tool}_{serial}" → {cid, start_ts}
        self._call_counter       = defaultdict(int)
        self.token_totals        = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        open(self.log_path, "w").close()  # fresh file per run

    def _write(self, record: dict):
        record.update({"run_id": self.run_id, "timestamp": time.time()})
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

    def on_tool_start(self, serialized, input_str, **kwargs):
        name   = serialized.get("name", "unknown")
        serial = self._call_counter[name]
        self._call_counter[name] += 1
        cid    = str(uuid.uuid4())
        self._pending[f"{name}_{serial}"] = {"cid": cid, "start_ts": time.perf_counter()}
        self._write({"event": "tool_start", "tool": name,
                     "input": str(input_str)[:300], "correlation_id": cid})

    def on_tool_end(self, output, **kwargs):
        # Determine which tool just finished
        name   = kwargs.get("name", "")
        if not name and self._call_counter:
            name = list(self._call_counter.keys())[-1]
        serial  = self._call_counter.get(name, 1) - 1
        pending = self._pending.get(f"{name}_{serial}", {})
        cid     = pending.get("cid", str(uuid.uuid4()))
        elapsed = round(time.perf_counter() - pending.get("start_ts", time.perf_counter()), 4)
        # Truncate large HTML in output to keep logs readable
        out_str = str(output)
        if len(out_str) > 400:
            out_str = out_str[:400] + "...[truncated]"
        self._write({"event": "tool_end", "tool": name,
                     "output": out_str, "elapsed_sec": elapsed, "correlation_id": cid})

    def on_agent_action(self, action, **kwargs):
        self._write({"event": "agent_action", "tool": action.tool,
                     "tool_input": str(action.tool_input)[:300],
                     "log": (action.log or "")[:300],
                     "correlation_id": str(uuid.uuid4())})

    def on_llm_start(self, serialized, prompts, **kwargs):
        self._write({"event": "llm_start",
                     "model": serialized.get("kwargs", {}).get("model_name", "unknown")})

    def on_llm_end(self, response, **kwargs):
        usage = {}
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage", {})
            for k in ("prompt_tokens", "completion_tokens", "total_tokens"):
                self.token_totals[k] += usage.get(k, 0)
        self._write({"event": "llm_end", "token_usage": usage})

print("ExecutionTraceHandler defined ✓")
print("Logs path: logs/execution_trace.jsonl")

## Step 4 — Agent Assembly

Builds a `create_openai_tools_agent` with an explicit tool-call order in the system prompt,
wrapped in `AgentExecutor` with `verbose=True` so you can see the reasoning chain live.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# ── LLM ─────────────────────────────────────────────────────
# Set OPENAI_API_KEY in your .env file, or uncomment the next line:
# os.environ["OPENAI_API_KEY"] = "sk-..."

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Prompt ───────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a price-comparison assistant. Always follow this exact sequence:\n"
     "1. Call fetch_html with Site A's URL\n"
     "2. Call extract_price on Site A's HTML\n"
     "3. Call fetch_html with Site B's URL\n"
     "4. Call extract_price on Site B's HTML\n"
     "5. Call compare_prices with both extracted prices\n"
     "Only fetch from approved domains (127.0.0.1:18080). "
     "Return the compare_prices result dict as your final answer."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# ── Agent + Executor ─────────────────────────────────────────
agent          = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=12)

print("AgentExecutor assembled ✓")
print(f"  Model  : {llm.model_name}")
print(f"  Tools  : {[t.name for t in tools]}")
print(f"  Max iter: {agent_executor.max_iterations}")

## Step 5 — Run A: Unmonitored Baseline

Runs the agent with only the execution-trace callback (no AgentTrust governance).
Check the JSONL for sane entries and timing before adding governance in Step 6.

In [ ]:
import time, uuid, json

SITE_A_URL = "http://127.0.0.1:18080/site-a/product"
SITE_B_URL = "http://127.0.0.1:18080/site-b/product"

run_id_a = str(uuid.uuid4())
handler_a = ExecutionTraceHandler(run_id=run_id_a)

print("=" * 62)
print("RUN A — Unmonitored Baseline")
print(f"Run ID  : {run_id_a}")
print("=" * 62)

wall_start = time.perf_counter()

try:
    result_a = agent_executor.invoke(
        {"input": (
            f"Compare the price of the widget on {SITE_A_URL} "
            f"and {SITE_B_URL}"
        )},
        config={"callbacks": [handler_a]},
    )
    wall_elapsed = round(time.perf_counter() - wall_start, 2)
    agent_answer = result_a.get("output", str(result_a))
except Exception as exc:
    wall_elapsed = round(time.perf_counter() - wall_start, 2)
    agent_answer = f"ERROR: {exc}"
    result_a = None

print("\n" + "=" * 62)
print("RUN A RESULT")
print("=" * 62)
print(f"  Answer        : {agent_answer}")
print(f"  Wall time     : {wall_elapsed}s")
print(f"  LLM tokens    : {handler_a.token_totals}")
print(f"\nExecution trace → logs/execution_trace.jsonl")

## Step 5b — Execution Trace Analysis

Loads `logs/execution_trace.jsonl` and prints per-phase metrics:
tool timings, event breakdown, and total LLM token usage.

In [ ]:
import json, pandas as pd
from IPython.display import display

def load_jsonl(path: str) -> pd.DataFrame:
    rows = []
    try:
        with open(path) as f:
            rows = [json.loads(l) for l in f if l.strip()]
    except FileNotFoundError:
        print(f"  (not found: {path})")
    return pd.DataFrame(rows) if rows else pd.DataFrame()

trace_a = load_jsonl("logs/execution_trace.jsonl")

print("=" * 62)
print("EXECUTION TRACE ANALYSIS — Run A")
print("=" * 62)

if trace_a.empty:
    print("  No trace data found — did Run A complete successfully?")
else:
    print(f"\nTotal events   : {len(trace_a)}")

    print("\nEvent breakdown:")
    print(trace_a["event"].value_counts().rename("count").to_string())

    tool_ends = trace_a[trace_a["event"] == "tool_end"].copy()
    if not tool_ends.empty:
        print("\nPer-tool timings:")
        cols = [c for c in ["tool","elapsed_sec","correlation_id"] if c in tool_ends.columns]
        print(tool_ends[cols].to_string(index=False))
        if "elapsed_sec" in tool_ends.columns:
            print(f"\n  Total tool time  : {tool_ends['elapsed_sec'].sum():.4f}s")
            print(f"  Slowest tool     : {tool_ends.loc[tool_ends['elapsed_sec'].idxmax(), 'tool']} ({tool_ends['elapsed_sec'].max():.4f}s)")

    llm_ends = trace_a[trace_a["event"] == "llm_end"]
    if not llm_ends.empty and "token_usage" in llm_ends.columns:
        print("\nLLM token usage per call:")
        for _, row in llm_ends.iterrows():
            print(f"  {row.get('token_usage', {})}")

    print("\nFull trace (all events):")
    view_cols = [c for c in ["event","tool","elapsed_sec","correlation_id","timestamp"] if c in trace_a.columns]
    display(trace_a[view_cols])

## Step 6 — Configure AgentTrust Policy

Writes `policy/price_agent.yaml` — five rules covering required output fields,
domain allow-listing, and a large-difference escalation trigger.

The embedded gateway picks up YAML files automatically.

In [ ]:
import os, yaml

os.makedirs("policy", exist_ok=True)

POLICY_YAML = """\
meta:
  id: price-agent
  version: "1.0"
  description: "Governance rules for the LangChain price-comparison agent"
  patterns:
    - "price-comparison-agent"

rules:
  - id: output_cheaper_site_present
    description: "Output must include a cheaper_site field"
    severity: high
    target: output.cheaper_site
    op: exists
    effect: deny

  - id: output_price_a_present
    description: "Output must include price_a"
    severity: high
    target: output.price_a
    op: exists
    effect: deny

  - id: output_price_b_present
    description: "Output must include price_b"
    severity: high
    target: output.price_b
    op: exists
    effect: deny

  - id: approved_domains_only
    description: "Only approved domains may be fetched"
    severity: critical
    target: output.domains_used
    op: not_in_list
    value: ["evil.com", "shadow-site.net"]
    effect: deny

  - id: large_price_difference_review
    description: "Differences above $50 are escalated for human review"
    severity: medium
    target: output.difference
    op: lte
    value: 50.0
    effect: review
"""

policy_path = "policy/price_agent.yaml"
with open(policy_path, "w") as f:
    f.write(POLICY_YAML)

loaded = yaml.safe_load(POLICY_YAML)
print(f"Policy written  : {policy_path}")
print(f"Rules           : {len(loaded['rules'])}")
print(f"Policy ID       : {loaded['meta']['id']}")
print(f"Version         : {loaded['meta']['version']}")
print()
print(f"  {'Rule ID':<35} {'Severity':>9}  {'Effect':>8}")
print(f"  {'-'*35}  {'-'*9}  {'-'*8}")
for rule in loaded["rules"]:
    print(f"  {rule['id']:<35} {rule['severity']:>9}  {rule['effect']:>8}")
print()
print("✓ Policy YAML valid and loaded")

## Step 6b — Validate Final Output Against AgentTrust

Uses `client.validate()` directly (Pattern C) to govern the final comparison dict
and display the full `ValidateResponse` — decision, risk tier, policy score, failures.

In [ ]:
from agentrust_sdk import AgentTrustClient, BlockedError

# Re-use the gateway started at the top of the notebook (Cell 2)
# If you restart the kernel, run Cell 2 first to re-start it.
client = AgentTrustClient()

# Build the structured output from our standalone comparison
final_output = {
    "cheaper_site": comparison["cheaper_site"],
    "price_a":      comparison["price_a"],
    "price_b":      comparison["price_b"],
    "difference":   comparison["difference"],
}

print(f"Validating: {final_output}")
print()

try:
    r = client.validate(
        agent_id="price-comparison-agent",
        user="qa-runner",
        input=f"Compare prices on {SITE_A_URL} and {SITE_B_URL}",
        output=final_output,
        framework="LangChain",
        metadata={
            "step": "final_output",
            "run_id": run_id_a,
            "policy_file": "policy/price_agent.yaml",
        },
    )

    print("=" * 62)
    print("AGENTRUST GOVERNANCE REPORT — Step 6")
    print("=" * 62)
    print(f"  envelope_id        : {r.envelope_id}")
    print(f"  approved           : {r.approved}")
    print(f"  blocked            : {r.blocked}")
    print(f"  needs_review       : {r.needs_review}")
    print()
    print(f"  decision.outcome   : {r.decision.outcome}")
    print(f"  decision.reason    : {r.decision.reason}")
    print(f"  policy_version     : {r.decision.policy_version}")
    print()
    print(f"  risk.tier          : {r.risk.tier}")
    print(f"  risk.score         : {r.risk.score}")
    print(f"  risk.reason        : {r.risk.reason}")
    print()
    print(f"  policy_score       : {r.validation.policy_score}")
    print(f"  final_confidence   : {r.validation.final_confidence}")
    print(f"  schema_score       : {r.validation.schema_score}")
    print(f"  tool_trust_score   : {r.validation.tool_trust_score}")
    print(f"  consistency_score  : {r.validation.consistency_score}")
    print(f"  failures           : {r.validation.failures}")

except BlockedError as e:
    print(f"BLOCKED by governance: {e.reason}  (envelope_id={e.envelope_id})")
except Exception as e:
    print(f"Governance note ({type(e).__name__}): {e}")
    print()
    print("  Tip: the embedded gateway requires AGENTRUST_KEY to be set.")
    print("  In dev mode it may return 401 and fail-open (governance skipped).")
    print("  The raw comparison result from Step 1c is still valid.")

## Overall Metrics Summary

Aggregates wall time, token usage, and governance outcome across all phases.

In [ ]:
import json

print("=" * 62)
print("END-TO-END METRICS SUMMARY")
print("=" * 62)

# ── Step 1 standalone phase metrics ─────────────────────────
print("\nStep 1 — Standalone Tool Phase Metrics:")
for tool, m in phase_metrics.items():
    print(f"  {tool:<22} calls={m['calls']}  duration={m['total_duration_sec']:.6f}s")

# ── Run A execution trace stats ──────────────────────────────
print("\nRun A — Agent Execution:")
print(f"  Wall time          : {wall_elapsed}s")
print(f"  LLM token totals   : {handler_a.token_totals}")
if not trace_a.empty:
    tool_ends_a = trace_a[trace_a["event"] == "tool_end"]
    if not tool_ends_a.empty and "elapsed_sec" in tool_ends_a.columns:
        print(f"  Total tool time    : {tool_ends_a['elapsed_sec'].sum():.4f}s")
        print(f"  LLM overhead       : {round(wall_elapsed - tool_ends_a['elapsed_sec'].sum(), 2)}s")
    print(f"  Agent actions      : {len(trace_a[trace_a['event'] == 'agent_action'])}")
    print(f"  LLM calls          : {len(trace_a[trace_a['event'] == 'llm_start'])}")

# ── Step 6 policy config ─────────────────────────────────────
print("\nStep 6 — Policy Configuration:")
print(f"  Policy file        : policy/price_agent.yaml")
print(f"  Rules loaded       : {len(loaded['rules'])}")
print(f"  Critical rules     : {sum(1 for r in loaded['rules'] if r['severity']=='critical')}")
print(f"  High rules         : {sum(1 for r in loaded['rules'] if r['severity']=='high')}")
print(f"  Escalation rules   : {sum(1 for r in loaded['rules'] if r['effect']=='review')}")

print("\n" + "=" * 62)
print("NEXT: Run Step 7 (run_b_agenttrust.py) to add per-tool governance")
print("      and compare governance_trace.jsonl against execution_trace.jsonl")
print("=" * 62)